In [1]:
import os
import cv2
import numpy as np

# Paths
yolo_base     = "/home/ubuntu/additional_drive/shwan_data/auto_annotations/temp_yolo/all_dataset_singleclass/"
images_train  = os.path.join(yolo_base, "images/train")
labels_train  = os.path.join(yolo_base, "labels/train")

sam_images   = "sam_dataset/images/train"
sam_masks    = "sam_dataset/masks/train"
os.makedirs(sam_images, exist_ok=True)
os.makedirs(sam_masks,  exist_ok=True)

# For each image + YOLO label, convert point annotation -> rough mask
# (Here we fill a small circle around each point as mask; you can refine later)
for img_name in os.listdir(images_train):
    if not img_name.lower().endswith((".jpg", ".png")):
        continue
    img_path = os.path.join(images_train, img_name)
    lbl_path = os.path.join(labels_train, os.path.splitext(img_name)[0] + ".txt")
    if not os.path.exists(lbl_path):
        continue

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Create blank mask
    mask = np.zeros((h, w), dtype=np.uint8)

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            # format: class x_center y_center [rest…]
            x_norm = float(parts[1])
            y_norm = float(parts[2])
            x_pix  = int(x_norm * w)
            y_pix  = int(y_norm * h)
            # draw circle radius 10px (rough)
            cv2.circle(mask, (x_pix, y_pix), radius=10, color=255, thickness=-1)

    # Save image and mask
    dst_img = os.path.join(sam_images, img_name)
    dst_mask= os.path.join(sam_masks, os.path.splitext(img_name)[0] + ".png")
    cv2.imwrite(dst_img, img)
    cv2.imwrite(dst_mask, mask)

print("✅ Dataset conversion done: images + roughly generated masks")


✅ Dataset conversion done: images + roughly generated masks
